# Task 2: Data Preprocessing for a Regression Problem

> **Note on data:** this notebook runs on `final_internship_sample.csv`, a 5,000-row random
> sample drawn from the full ~12,287-row dataset *before* any Task 1 cleaning was applied
> (so it still contains the same nulls, negative fares, zero-passenger rows, and corrupted
> GPS/distance outliers that Task 1 identified). The pipeline logic below is identical to what
> you'd run on the full file - only the row count differs.
>This was used to ease the process of reuploading the csv file to Jupiter Notebook everytime working on the task.

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split, KFold, cross_val_score, RandomizedSearchCV
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from trip_transformer import TripFeatureEngineer, RAW_FEATURE_COLUMNS

RANDOM_STATE = 42
pd.set_option('display.max_columns', 30)

df = pd.read_csv('final_internship_sample.csv')
print('Raw shape:', df.shape)
df.head()


Raw shape: (5000, 26)


,User ID,User Name,Driver Name,Car Condition,Weather,Traffic Condition,key,fare_amount,pickup_datetime,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,passenger_count,hour,day,month,weekday,year,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing
0,nVe9j57F,Megan Perez,Jose Strickland,Good,sunny,Congested Traffic,2009-07-29 18:05:51.0000002,27.7,2009-07-29 18:05:51,-1.287764,0.709451,-1.289220,0.711524,1,18,29,7,2,2009,17.063978,59.603039,17.369454,39.784692,33.324768,14.968782,0.488822
1,Mw46dBzu,April Mills,Matthew Walters,Bad,rainy,Flow Traffic,2014-04-21 08:43:00.00000051,7.5,2014-04-21 08:43:00,-1.290810,0.711731,-1.290740,0.711506,1,8,21,4,0,2014,42.037743,40.059320,14.148673,23.859636,15.592287,1.472966,-2.908325
2,LabUz9GM,Grant Collier,Jason Kent,Good,cloudy,Dense Traffic,2011-07-25 12:20:37.0000002,5.3,2011-07-25 12:20:37,-1.290917,0.711487,-1.290699,0.711780,1,12,25,7,0,2011,42.437751,39.851693,14.491780,23.783006,15.543184,2.144215,-0.513050
3,ehcYW3w2,Christopher Powell,Nicole Coffey,Excellent,stormy,Dense Traffic,2013-07-03 14:51:48.0000001,18.0,2013-07-03 14:51:48,0.000000,0.000000,0.000000,0.000000,1,14,3,7,2,2013,17293.486312,17360.264559,17314.695444,17339.554005,17334.256605,0.000000,0.000000
4,5Yrv0t3g,Rickey Carr,Christine Santos,Very Good,stormy,Dense Traffic,2011-03-20 17:10:38.0000001,34.5,2011-03-20 17:10:38,-1.291299,0.711377,-1.289323,0.711645,2,17,20,3,6,2011,38.842946,43.604393,10.093617,26.475673,18.355914,9.683302,-1.393111


## Handling Missing Values

In [6]:
missing_pct = df.isnull().mean().sort_values(ascending=False) * 100
print(missing_pct[missing_pct > 0])
print('\nTotal rows with at least one missing value:', df.isnull().any(axis=1).sum(), '/', len(df))

Series([], dtype: float64)

Total rows with at least one missing value: 0 / 5000


In [7]:
df = df.dropna()
print('Shape after dropping missing rows:', df.shape)

Shape after dropping missing rows: (5000, 26)


##  Detecting and Removing Duplicates

In [8]:
exact_dupes = df.duplicated().sum()
key_dupes = df.duplicated(subset=['key']).sum()
user_id_dupes = df.duplicated(subset=['User ID']).sum()
print('Exact duplicate rows:', exact_dupes)
print('Duplicate ride keys:', key_dupes)
print('Duplicate User IDs:', user_id_dupes)

Exact duplicate rows: 0
Duplicate ride keys: 0
Duplicate User IDs: 0


## Fixing Data-Entry Errors (from Task 1)

Task 1's EDA surfaced several **impossible, not just unusual** values that are data-entry/collection
errors rather than rare-but-real trips (per the "is this outlier a mistake or a rare truth?" rule
in Section 7 of the brief). These need to be fixed *before* feature engineering and splitting,
because they corrupt the engineered distance/location features too.

In [9]:
print('fare_amount <= 0:', (df['fare_amount'] <= 0).sum())
print('passenger_count == 0:', (df['passenger_count'] == 0).sum())
print('distance == 0:', (df['distance'] == 0).sum())
print('distance > 500 km:', (df['distance'] > 500).sum())

coord_bad = ~((df['pickup_longitude'] < 0) & (df['dropoff_longitude'] < 0) &
              (df['pickup_latitude'] > 0) & (df['dropoff_latitude'] > 0))
print('Rows with sign-flipped/corrupted coordinates:', coord_bad.sum())

fare_amount <= 0: 0
passenger_count == 0: 21
distance == 0: 152
distance > 500 km: 4
Rows with sign-flipped/corrupted coordinates: 105


In [10]:
before = len(df)
df = df[df['fare_amount'] > 0]
df = df[df['passenger_count'] > 0]
df = df[(df['distance'] > 0) & (df['distance'] < 500)]
coord_ok = (df['pickup_longitude'] < 0) & (df['dropoff_longitude'] < 0) & \
           (df['pickup_latitude'] > 0) & (df['dropoff_latitude'] > 0)
df = df[coord_ok]
print(f'Dropped {before - len(df)} data-error rows -> new shape: {df.shape}')


Dropped 181 data-error rows -> new shape: (4819, 26)


### Dropping redundant / non-predictive columns

- `User ID`, `User Name`, `Driver Name`, `key`: unique identifiers — dropped
- `pickup_datetime`, `day`: redundant once we keep `hour` / `weekday` / `month` / `year`
- Precomputed `distance`, `bearing`, and landmark distances are also dropped here:
  they are **recomputed inside the sklearn pipeline** by `TripFeatureEngineer`
  from the raw coordinates, so training and the Flask app share one path


In [11]:
corr_check = df[['distance', 'bearing', 'pickup_longitude', 'pickup_latitude',
                  'dropoff_longitude', 'dropoff_latitude', 'fare_amount']].corr()['fare_amount']
print(corr_check.sort_values(ascending=False))

fare_amount          1.000000
distance             0.768487
pickup_longitude     0.410149
dropoff_longitude    0.264781
bearing             -0.015754
pickup_latitude     -0.122643
dropoff_latitude    -0.170059
Name: fare_amount, dtype: float64


Raw coordinates correlate far more weakly with `fare_amount` than the engineered `distance`
does, confirming `distance`/`bearing`/landmark-distance features are the more useful,
lower-redundancy representation of location.

In [12]:
drop_cols = [
    'User ID', 'User Name', 'Driver Name', 'key', 'pickup_datetime', 'day',
    # Recomputed inside TripFeatureEngineer from coordinates / hour / weekday:
    'distance', 'bearing', 'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist',
]
df = df.drop(columns=drop_cols)
print('Remaining columns:', list(df.columns))


Remaining columns: ['Car Condition', 'Weather', 'Traffic Condition', 'fare_amount', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude', 'passenger_count', 'hour', 'month', 'weekday', 'year']


## Train/Test Split and Avoiding Data Leakage

The dataset is i.i.d. tabular data, so a random split is appropriate here
rather than a chronological one.

In [13]:
y = df['fare_amount']
X = df[RAW_FEATURE_COLUMNS].copy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print('Train shape:', X_train.shape, '| Test shape:', X_test.shape)
print('Raw feature columns fed to the pipeline:', list(X.columns))


Train shape: (3855, 12) | Test shape: (964, 12)
Raw feature columns fed to the pipeline: ['Car Condition', 'Weather', 'Traffic Condition', 'passenger_count', 'month', 'year', 'hour', 'weekday', 'pickup_latitude', 'pickup_longitude', 'dropoff_latitude', 'dropoff_longitude']


## Trip feature engineering inside the pipeline

This step used to happen in plain pandas *before* the model pipeline. That made
deployment awkward: the Flask app would have to repeat the same formulas by hand.

Instead, `TripFeatureEngineer` is the **first step** of the sklearn `Pipeline`.
Given raw trip columns (coordinates in radians, `hour`, `weekday`, categoricals, …)
it builds the derived features the rest of the pipeline expects:

- haversine `distance` and trip `bearing`
- landmark distances (`jfk_dist`, `ewr_dist`, `lga_dist`, `sol_dist`, `nyc_dist`)
- cyclic time encodings (`hour_sin` / `hour_cos`, `weekday_sin` / `weekday_cos`)
- flags `is_weekend` and `is_airport_trip`

Because this lives inside the fitted pipeline object, `joblib.dump` saves it with
the encoders, scaler, feature selector, and model — so training and serving share
one artifact (`final_pipeline.joblib`).


In [14]:
# First pipeline step: raw trip columns -> engineered model features
# (class definition lives in trip_transformer.py so joblib can load it in Flask)
feature_engineer = TripFeatureEngineer()

# Quick sanity check on one training row
_sample_eng = feature_engineer.fit_transform(X_train.iloc[[0]])
print('Engineered columns:', list(_sample_eng.columns))
_sample_eng.T


Engineered columns: ['Car Condition', 'Weather', 'Traffic Condition', 'passenger_count', 'month', 'year', 'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist', 'distance', 'bearing', 'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos', 'is_weekend', 'is_airport_trip']


,0
Car Condition,Good
Weather,windy
Traffic Condition,Congested Traffic
passenger_count,1.0
month,10.0
year,2009.0
jfk_dist,42.875353
ewr_dist,36.935284
lga_dist,16.950716
sol_dist,20.851589


## Encoding Categorical Variables & Feature Scaling

After `TripFeatureEngineer`, three categorical columns remain: `Car Condition`,
`Traffic Condition`, `Weather`.

Skewed distance / landmark columns get `log1p` then `StandardScaler`. Other
numerics get a plain `StandardScaler`. Binary flags pass through unscaled.

All of this is wrapped in a `ColumnTransformer` that runs **after** engineering,
inside the same `Pipeline`.


In [15]:
ordinal_cols = ['Car Condition', 'Traffic Condition']
ordinal_categories = [['Bad', 'Good', 'Very Good', 'Excellent'],
                      ['Flow Traffic', 'Dense Traffic', 'Congested Traffic']]
onehot_cols = ['Weather']
skewed_numeric = ['distance', 'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist']
other_numeric = ['passenger_count', 'month', 'year', 'bearing',
                  'hour_sin', 'hour_cos', 'weekday_sin', 'weekday_cos']
binary_passthrough = ['is_weekend', 'is_airport_trip']

log1p_transformer = FunctionTransformer(np.log1p, feature_names_out='one-to-one')

preprocessor = ColumnTransformer(transformers=[
    ('ord', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encode', OrdinalEncoder(categories=ordinal_categories)),
        ('scale', StandardScaler())
    ]), ordinal_cols),
    ('onehot', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'))
    ]), onehot_cols),
    ('skewed', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('log', log1p_transformer),
        ('scale', StandardScaler())
    ]), skewed_numeric),
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), other_numeric),
    ('bin', 'passthrough', binary_passthrough),
])
preprocessor


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ord', ...), ('onehot', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featur

## Feature Selection

Lasso's L1 penalty automatically shrinks the coefficients of
features that add little predictive value (after multicollinearity was already reduced in
Section 4 by dropping raw coordinates) to zero; `threshold='median'` keeps the stronger half of
the remaining engineered features.

## Cross-Validation & Baseline Model
The baseline model is plain `LinearRegression` - simple, fast, and a fair reference point before
trying anything more complex.

In [16]:
def make_model_pipeline(model):
    """Full deployable pipeline: engineer -> preprocess -> select -> model."""
    return Pipeline([
        ('engineer', TripFeatureEngineer()),
        ('preprocess', preprocessor),
        ('select', SelectFromModel(
            Lasso(alpha=0.01, max_iter=20000, random_state=RANDOM_STATE),
            threshold='median')),
        ('model', model),
    ])

baseline_pipeline = make_model_pipeline(LinearRegression())
baseline_model = TransformedTargetRegressor(
    regressor=baseline_pipeline, func=np.log1p, inverse_func=np.expm1)

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores = cross_val_score(baseline_model, X_train, y_train, cv=kf,
                             scoring='neg_root_mean_squared_error')
print(f'Baseline 5-fold CV RMSE: {-cv_scores.mean():.3f} (+/- {cv_scores.std():.3f})')

baseline_model.fit(X_train, y_train)
pred_base = baseline_model.predict(X_test)
mae_b = mean_absolute_error(y_test, pred_base)
rmse_b = mean_squared_error(y_test, pred_base) ** 0.5
r2_b = r2_score(y_test, pred_base)
print(f'Baseline test set -> MAE: {mae_b:.3f} | RMSE: {rmse_b:.3f} | R2: {r2_b:.3f}')


Baseline 5-fold CV RMSE: 4.491 (+/- 0.777)
Baseline test set -> MAE: 2.027 | RMSE: 4.177 | R2: 0.833


## Hyperparameter Tuning

For the tuned model we switch to `RandomForestRegressor`. `RandomizedSearchCV` is used
instead of an exhaustive grid since the hyperparameter space has too many combinations to try all of.

In [17]:
tuned_pipeline = make_model_pipeline(RandomForestRegressor(random_state=RANDOM_STATE))
tuned_model = TransformedTargetRegressor(
    regressor=tuned_pipeline, func=np.log1p, inverse_func=np.expm1)

param_dist = {
    'regressor__model__n_estimators': [100, 200, 300, 400],
    'regressor__model__max_depth': [None, 5, 10, 15, 20],
    'regressor__model__min_samples_split': [2, 5, 10],
    'regressor__model__max_features': ['sqrt', 'log2', 0.5, 1.0],
}
search = RandomizedSearchCV(tuned_model, param_distributions=param_dist, n_iter=15, cv=kf,
                             scoring='neg_root_mean_squared_error', random_state=RANDOM_STATE,
                             n_jobs=-1)
search.fit(X_train, y_train)
print('Best params:', search.best_params_)
print(f'Best CV RMSE: {-search.best_score_:.3f}')


Best params: {'regressor__model__n_estimators': 200, 'regressor__model__min_samples_split': 5, 'regressor__model__max_features': 1.0, 'regressor__model__max_depth': 20}
Best CV RMSE: 4.123


In [18]:
best_model = search.best_estimator_
pred_tuned = best_model.predict(X_test)
mae_t = mean_absolute_error(y_test, pred_tuned)
rmse_t = mean_squared_error(y_test, pred_tuned) ** 0.5
r2_t = r2_score(y_test, pred_tuned)
print(f'Tuned test set -> MAE: {mae_t:.3f} | RMSE: {rmse_t:.3f} | R2: {r2_t:.3f}')


Tuned test set -> MAE: 1.913 | RMSE: 4.167 | R2: 0.834


## Task 3 Part A — Additional Models

Task 3 asks for at least three regression models. In addition to Linear Regression and
the tuned Random Forest above, we also evaluate a Decision Tree and Gradient Boosting
regressor on the same preprocessed features and report MAE / RMSE / R² on the test set.


In [19]:
def make_wrapped(model):
    return TransformedTargetRegressor(
        regressor=make_model_pipeline(model),
        func=np.log1p, inverse_func=np.expm1)

dt_model = make_wrapped(DecisionTreeRegressor(
    random_state=RANDOM_STATE, max_depth=12, min_samples_leaf=5))
dt_model.fit(X_train, y_train)
pred_dt = dt_model.predict(X_test)
mae_dt = mean_absolute_error(y_test, pred_dt)
rmse_dt = mean_squared_error(y_test, pred_dt) ** 0.5
r2_dt = r2_score(y_test, pred_dt)
print(f'Decision Tree test set -> MAE: {mae_dt:.3f} | RMSE: {rmse_dt:.3f} | R2: {r2_dt:.3f}')

gb_model = make_wrapped(GradientBoostingRegressor(random_state=RANDOM_STATE))
gb_model.fit(X_train, y_train)
pred_gb = gb_model.predict(X_test)
mae_gb = mean_absolute_error(y_test, pred_gb)
rmse_gb = mean_squared_error(y_test, pred_gb) ** 0.5
r2_gb = r2_score(y_test, pred_gb)
print(f'Gradient Boosting test set -> MAE: {mae_gb:.3f} | RMSE: {rmse_gb:.3f} | R2: {r2_gb:.3f}')


Decision Tree test set -> MAE: 2.321 | RMSE: 4.966 | R2: 0.764
Gradient Boosting test set -> MAE: 1.838 | RMSE: 3.920 | R2: 0.853


## Results: Model Comparison (Task 3 Part A)

Test-set metrics for all candidates. The deployed model is chosen from these numbers
(not from training error alone).


In [20]:
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Decision Tree', 'Tuned Random Forest', 'Gradient Boosting'],
    'MAE': [mae_b, mae_dt, mae_t, mae_gb],
    'RMSE': [rmse_b, rmse_dt, rmse_t, rmse_gb],
    'R2': [r2_b, r2_dt, r2_t, r2_gb]
}).sort_values('RMSE')
results


,Model,MAE,RMSE,R2
3,Gradient Boosting,1.837843,3.919503,0.852851
2,Tuned Random Forest,1.913161,4.166599,0.833713
0,Linear Regression,2.027431,4.177431,0.832847
1,Decision Tree,2.321416,4.966106,0.763774


### Justification for the deployed model (Task 3 Part A)

We select **Gradient Boosting** for deployment when it leads on test MAE / RMSE / R²
(see the comparison table and `train_and_save.py`). It consistently beats the linear
baseline and the single Decision Tree, and matches or exceeds the tuned Random Forest
on held-out error while remaining a single sklearn estimator inside the same
preprocess → Lasso select → model pipeline. Prediction latency is acceptable for a
local Flask demo. The entire fitted `TransformedTargetRegressor` pipeline is saved so
encoding, scaling, feature selection, and the log1p target transform are applied
identically at inference time in the web app.


## Saving the Final Pipeline

In [21]:
# Persist the best test-set model (Gradient Boosting).
# The saved object includes TripFeatureEngineer + preprocess + select + model.
best_model = gb_model
joblib.dump(best_model, 'final_pipeline.joblib')
print('Saved fitted pipeline to final_pipeline.joblib')
print('Pipeline steps:', list(best_model.regressor_.named_steps.keys()))


Saved fitted pipeline to final_pipeline.joblib
Pipeline steps: ['engineer', 'preprocess', 'select', 'model']


## What I Would Try Next

- I will try to optimize both of the algorithms used with tuning the hyperparameters
- I will test using other data sets
- I will be using different data distribution functions for the validation other than the K fold and retest my results
- I will try to manually filter the used columns or try another value in the Lasso function other than the median